In [9]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential

load_dotenv(override=True)

# Two clients this time — one for DI, one for LLM
openai_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION")
)

di_client = DocumentIntelligenceClient(
    endpoint=os.getenv("DI_ENDPOINT"),
    credential=AzureKeyCredential(os.getenv("DI_KEY"))
)

print("Both clients ready ✅")

Both clients ready ✅


In [10]:
from azure.ai.documentintelligence.models import DocumentAnalysisFeature

def extract_with_di(file_path: str) -> str:
    with open(file_path, "rb") as f:
        poller = di_client.begin_analyze_document(
            model_id="prebuilt-layout",     # latest recommended
            body=f,
            features=[DocumentAnalysisFeature.KEY_VALUE_PAIRS]  # replaces prebuilt-document
        )
    
    result = poller.result()
    
    # result.content = all text in reading order
    # this is what we send to LLM
    print(f"Pages: {len(result.pages)}")
    print(f"Raw text length: {len(result.content)} chars")
    print("\n--- Preview first 500 chars ---")
    print(result.content[:500])
    
    return result.content

# test on one file first
raw_text = extract_with_di("pdfs/24.12.16 HJK Dalies.pdf")

Pages: 2
Raw text length: 1842 chars

--- Preview first 500 chars ---
The Whiting-Turner Contracting Company Job Name: 019985 Sinai Cancer Center Job Address: 2401 Belvedere Ave 21215 WT Job #: 019985
Contractor:
H.J.K masonry
WT 100 YEARS
Contractor Daily Work Report
DAY/DATE:
12.17.24
Shift Start Time:
6:30
Shift Complete Time:
3:00
Workers Onsite
Classification:
Number:
Hours Worked
Notes:
Foreman
2
8 cg
Bricklayers
2
8 cg
Total Manhours Worked:
32
Describe Work Performed: ( Location, activity, etc. ) Setting up control along 4.9 between columns & A and N.
Majo


In [11]:
from pydantic import BaseModel

class DailyReport(BaseModel):
    reasoning: str
    Vendor: str
    Date: str           # YYYY-MM-DD
    TotalWorkers: int
    TotalWorkHr: float

class DailyReportList(BaseModel):
    reports: list[DailyReport]  # one per page/report in the doc

def structure_with_llm(raw_text: str) -> DailyReportList:
    response = openai_client.beta.chat.completions.parse(
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
        messages=[
            {
                "role": "system",
                "content": "You extract structured data from contractor daily work report text."
            },
            {
                "role": "user",
                "content": f"""Below is text extracted from a contractor daily work report document.
It may contain multiple pages/reports.

This PDF has multiple pages. Each page is one daily report or multiple pages might have information spread across. Your work is to analyse whole file and give accurate info on below fields
Extract one record per page:
- Vendor: contractor company name from whom WHiting Turner is taking service from
- Date: YYYY-MM-DD. Year is always greater then 2000
- TotalWorkers: SUM of the Number column across all rows on that page
- TotalWorkHr: use printed Total Manhours if filled, else sum hours x workers
First write your reasoning, then extract values

Document text:
{raw_text}"""
            }
        ],
        response_format=DailyReportList,
        temperature=0
    )

    print(f"Tokens — input: {response.usage.prompt_tokens}, output: {response.usage.completion_tokens}")
    return response.choices[0].message.parsed

# run it
result = structure_with_llm(raw_text)

for r in result.reports:
    print(r)

Tokens — input: 824, output: 232
reasoning="Page 1: Vendor is 'H.J.K masonry' as stated under Contractor. Date is given as 12.17.24 which corresponds to 2024-12-17. TotalWorkers is sum of Number column: Foreman 2 + Bricklayers 2 = 4 workers. TotalWorkHr is given as Total Manhours Worked: 32, so use 32 directly." Vendor='H.J.K masonry' Date='2024-12-17' TotalWorkers=4 TotalWorkHr=32.0
reasoning="Page 2: Vendor is 'H. J.K masonry' (same as page 1, slight spacing difference). Date is 12.18.24 which corresponds to 2024-12-18. TotalWorkers is sum of Number column: Foreman 1 + Bricklayer 1 + Operator 7 = 9 workers. Total Manhours Worked is given as 24, so use 24 directly." Vendor='H. J.K masonry' Date='2024-12-18' TotalWorkers=9 TotalWorkHr=24.0


In [12]:
print(raw_text)

The Whiting-Turner Contracting Company Job Name: 019985 Sinai Cancer Center Job Address: 2401 Belvedere Ave 21215 WT Job #: 019985
Contractor:
H.J.K masonry
WT 100 YEARS
Contractor Daily Work Report
DAY/DATE:
12.17.24
Shift Start Time:
6:30
Shift Complete Time:
3:00
Workers Onsite
Classification:
Number:
Hours Worked
Notes:
Foreman
2
8 cg
Bricklayers
2
8 cg
Total Manhours Worked:
32
Describe Work Performed: ( Location, activity, etc. ) Setting up control along 4.9 between columns & A and N.
Major Equipment Onsite (i.e. - Lifts, earthmoving equip., welder, swingstage)
Quantity:
Equipment Description:
N/A
Safety
Did any accidents or near misses occur?
Yes
X :selected:
No
If yes was a report filed with W-T (who? What? How?)? Yes
No N/A
Major Material Deliveries
Inspections/ Tests (Area?, description?, witness?)
Signature:
Print Name: J. Mueller :unselected: :unselected:
The Whiting-Turner Contracting Company Job Name: 019985 Sinai Cancer Center Job Address: 2401 Belvedere Ave 21215 WT Job

In [13]:
from pathlib import Path
Path(file).suffix.lower()  
# .pdf → send as pdf
# .png/.jpg/.jpeg → send as image
# content_type changes accordingly

'.pdf'

In [14]:
import os, json, base64
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentAnalysisFeature
from azure.core.credentials import AzureKeyCredential
from pydantic import BaseModel

load_dotenv(override=True)

openai_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION")
)

di_client = DocumentIntelligenceClient(
    endpoint=os.getenv("DI_ENDPOINT"),
    credential=AzureKeyCredential(os.getenv("DI_KEY"))
)

class DailyReport(BaseModel):
    reasoning: str
    Vendor: str
    Date: str
    TotalWorkers: int
    TotalWorkHr: float

class DailyReportList(BaseModel):
    reports: list[DailyReport]

print("Ready ✅")

Ready ✅


In [15]:
# ---- Cost per 1000 tokens (Azure OpenAI gpt-4.1-mini) ----
# Source: https://azure.microsoft.com/pricing/details/cognitive-services/openai-service/
COST_INPUT_PER_1K  = 0.00040   # $0.0004 per 1K input tokens
COST_OUTPUT_PER_1K = 0.00160   # $0.0016 per 1K output tokens

# ---- DI cost ----
# prebuilt-layout = $10 per 1000 pages = $0.01 per page
DI_COST_PER_PAGE = 0.01

def calculate_cost(input_tokens: int, output_tokens: int, di_pages: int = 0) -> dict:
    llm_cost = (input_tokens / 1000 * COST_INPUT_PER_1K) + \
               (output_tokens / 1000 * COST_OUTPUT_PER_1K)
    di_cost  = di_pages * DI_COST_PER_PAGE
    return {
        "llm_cost_usd":   round(llm_cost, 6),
        "di_cost_usd":    round(di_cost, 6),
        "total_cost_usd": round(llm_cost + di_cost, 6)
    }

print("Cost config ready ✅")
print(f"LLM input:  ${COST_INPUT_PER_1K}/1K tokens")
print(f"LLM output: ${COST_OUTPUT_PER_1K}/1K tokens")
print(f"DI layout:  ${DI_COST_PER_PAGE}/page")

Cost config ready ✅
LLM input:  $0.0004/1K tokens
LLM output: $0.0016/1K tokens
DI layout:  $0.01/page


In [16]:
# map file extension to content type DI expects
CONTENT_TYPE_MAP = {
    ".pdf":  "application/pdf",
    ".png":  "image/png",
    ".jpg":  "image/jpeg",
    ".jpeg": "image/jpeg",
    ".tiff": "image/tiff"
}

def extract_with_di(file_path: str) -> dict:
    """
    Send any supported file to Azure DI
    Returns raw text + page count
    """
    path = Path(file_path)
    ext  = path.suffix.lower()
    
    # check if we support this format
    if ext not in CONTENT_TYPE_MAP:
        raise ValueError(f"Unsupported format: {ext}")
    
    content_type = CONTENT_TYPE_MAP[ext]
    
    with open(file_path, "rb") as f:
        poller = di_client.begin_analyze_document(
            model_id="prebuilt-layout",
            body=f,
            content_type=content_type,
            features=[DocumentAnalysisFeature.KEY_VALUE_PAIRS]
        )
    
    result = poller.result()
    
    return {
        "raw_text":  result.content,
        "pages":     len(result.pages)
    }

print("DI function ready ✅")

DI function ready ✅


In [17]:
# test on one file — use any file from your pdfimgs folder
test = extract_with_di("pdfimgs/24.12.16 HJK Dalies.pdf")

print(f"Pages: {test['pages']}")
print(f"Text length: {len(test['raw_text'])} chars")
print(f"Preview:\n{test['raw_text'][:300]}")

Pages: 2
Text length: 1842 chars
Preview:
The Whiting-Turner Contracting Company Job Name: 019985 Sinai Cancer Center Job Address: 2401 Belvedere Ave 21215 WT Job #: 019985
Contractor:
H.J.K masonry
WT 100 YEARS
Contractor Daily Work Report
DAY/DATE:
12.17.24
Shift Start Time:
6:30
Shift Complete Time:
3:00
Workers Onsite
Classification:
Nu


In [18]:
def structure_with_llm(raw_text: str) -> tuple[DailyReportList, dict]:
    """
    Takes DI raw text → sends to LLM → returns structured reports + token info
    """
    response = openai_client.beta.chat.completions.parse(
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
        messages=[
            {
                "role": "system",
                "content": "You extract structured data from contractor daily work report text."
            },
            {
                "role": "user",
                "content": f"""Below is text extracted from a contractor daily work report.
It may contain multiple pages/reports.

This PDF has multiple pages. Each page is one daily report or multiple pages might have information spread across. Your work is to analyse whole file and give accurate info on below fields
Extract one record per page:
- Vendor: contractor company name from whom WHiting Turner is taking service from
- Date: YYYY-MM-DD. Year is always greater then 2000
- TotalWorkers: SUM of the Number column across all rows on that page
- TotalWorkHr: use printed Total Manhours if filled, else sum hours x workers
First write your reasoning, then extract values

Document text:
{raw_text}"""
            }
        ],
        response_format=DailyReportList,
        temperature=0
    )

    usage = response.usage
    token_info = {
        "input_tokens":  usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens":  usage.total_tokens
    }

    return response.choices[0].message.parsed, token_info

print("LLM function ready ✅")

LLM function ready ✅


In [19]:
def process_file(file_path: str) -> dict:
    """
    Full pipeline for one file:
    File → DI → LLM → structured output + cost
    """
    path = Path(file_path)
    print(f"\nProcessing: {path.name}")

    # Step 1 — DI extraction
    di_output   = extract_with_di(file_path)
    raw_text    = di_output["raw_text"]
    pages       = di_output["pages"]
    print(f"  DI done — pages: {pages}, text: {len(raw_text)} chars")

    # Step 2 — LLM structuring
    result, token_info = structure_with_llm(raw_text)
    print(f"  LLM done — reports: {len(result.reports)}, tokens: {token_info['total_tokens']}")

    # Step 3 — cost calculation
    cost = calculate_cost(
        input_tokens  = token_info["input_tokens"],
        output_tokens = token_info["output_tokens"],
        di_pages      = pages
    )
    print(f"  Cost: ${cost['total_cost_usd']}")

    return {
        "file":         path.name,
        "format":       path.suffix.lower(),
        "pages":        pages,
        "reports":      result.model_dump(
                            exclude={"reports": {"__all__": {"reasoning"}}}
                        )["reports"],
        "reasoning":    [r.reasoning for r in result.reports],  # audit trail
        "tokens":       token_info,
        "cost":         cost,
        "error":        None
    }

print("Batch function ready ✅")

Batch function ready ✅


In [20]:
from pathlib import Path


FOLDER = "pdfimgs"

all_files = [
    f for f in Path(FOLDER).iterdir()
    if f.suffix.lower() in [".pdf", ".png", ".jpg", ".jpeg"]
]

print(f"Found {len(all_files)} files to process")

results  = []
errors   = []

for file in all_files:
    try:
        result = process_file(str(file))
        results.append(result)
    except Exception as e:
        print(f"  ❌ Error: {e}")
        errors.append({"file": file.name, "error": str(e)})

# save results
with open("di_results.json", "w") as f:
    json.dump({"results": results, "errors": errors}, f, indent=2)

print(f"\n✅ Done — {len(results)} processed, {len(errors)} errors")

Found 29 files to process

Processing: 1.20.25-1.24.25.pdf
  DI done — pages: 2, text: 2164 chars
  LLM done — reports: 2, tokens: 1174
  Cost: $0.020768

Processing: 2.17.25-2.18.25 Dailys.pdf
  DI done — pages: 2, text: 1684 chars
  LLM done — reports: 2, tokens: 1074
  Cost: $0.020775

Processing: 24.12.16 HJK Dalies.pdf
  DI done — pages: 2, text: 1842 chars
  LLM done — reports: 2, tokens: 1011
  Cost: $0.02063

Processing: 3.12.26 - Daily Report.pdf
  DI done — pages: 2, text: 2219 chars
  LLM done — reports: 1, tokens: 1306
  Cost: $0.020689

Processing: CCC - WT Daily UD ECUUP (4).pdf
  DI done — pages: 1, text: 879 chars
  LLM done — reports: 1, tokens: 759
  Cost: $0.010576

Processing: Daily Job Diary 1-23-25.pdf
  DI done — pages: 1, text: 1368 chars
  LLM done — reports: 1, tokens: 811
  Cost: $0.010501

Processing: Daily Log - 03-12-2026 - MIH Tower & CUP (1).pdf
  DI done — pages: 2, text: 1194 chars
  LLM done — reports: 1, tokens: 828
  Cost: $0.020527

Processing: DAI

In [21]:
print(f"\n{'File':<40} {'Fmt':<6} {'Pg':<4} {'Reports':<8} {'Tokens':<8} {'Cost $':<10}")
print("-" * 80)

total_tokens = 0
total_cost   = 0

for r in results:
    print(f"{r['file']:<40} {r['format']:<6} {r['pages']:<4} "
          f"{len(r['reports']):<8} {r['tokens']['total_tokens']:<8} "
          f"${r['cost']['total_cost_usd']:<10}")
    total_tokens += r["tokens"]["total_tokens"]
    total_cost   += r["cost"]["total_cost_usd"]

print("-" * 80)
print(f"{'TOTAL':<40} {'':6} {'':4} {'':8} {total_tokens:<8} ${round(total_cost,6)}")

if errors:
    print(f"\n❌ Errors ({len(errors)}):")
    for e in errors:
        print(f"  {e['file']}: {e['error']}")


File                                     Fmt    Pg   Reports  Tokens   Cost $    
--------------------------------------------------------------------------------
1.20.25-1.24.25.pdf                      .pdf   2    2        1174     $0.020768  
2.17.25-2.18.25 Dailys.pdf               .pdf   2    2        1074     $0.020775  
24.12.16 HJK Dalies.pdf                  .pdf   2    2        1011     $0.02063   
3.12.26 - Daily Report.pdf               .pdf   2    1        1306     $0.020689  
CCC - WT Daily UD ECUUP (4).pdf          .pdf   1    1        759      $0.010576  
Daily Job Diary 1-23-25.pdf              .pdf   1    1        811      $0.010501  
Daily Log - 03-12-2026 - MIH Tower & CUP (1).pdf .pdf   2    1        828      $0.020527  
DAILY REPORT NOVEN - Sub Example.pdf     .pdf   1    1        882      $0.010535  
DR_12192025 (1).pdf                      .pdf   1    1        687      $0.010428  
f39f9ee9-980d-47c2-a70a-87290fab69d0.pdf .pdf   2    1        847      $0.020526 